# ✅ Functional Test: Snapshot Metadata Generation
This notebook runs a real experiment and verifies that the resulting `.meta.json` snapshot contains expected fields and values.

In [1]:
from pathlib import Path
import os, sys
from dotenv import load_dotenv

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_score_providers import seed_score_providers
from app.db.seeders.seed_tool_providers import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.seeders.seed_agent_providers import seed_agent_providers
from app.db.seeders.seed_state_providers import seed_state_providers
from app.db.seeders.seed_system_providers import seed_system_providers
from app.db.seeders.seed_controller_providers import seed_controller_providers
from app.db.seeders.seed_program_provider import seed_program_providers
from app.db.seeders.seed_session_configs import seed_session_configs
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)
    seed_agent_providers(session)
    seed_state_providers(session)
    seed_system_providers(session)
    seed_controller_providers(session)
    seed_program_providers(session)
    seed_session_configs(session)

# Load from env/.env relative to project root
dotenv_path = Path("env/.env").resolve()
if dotenv_path.exists():
    load_dotenv(dotenv_path)
    print("✅ Loaded environment variables from env/.env")
else:
    raise FileNotFoundError(f"❌ Missing .env file at: {dotenv_path}")

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt 'generate' with ID: 1 and GUID: 73f54440-ae03-4450-8a34-85e487b81b1e
Seeded AgentPrompt 'linting_generator_agent' with ID: 2 and GUID: 41682fed-01a9-4b90-b631-ae091c7437f7
Seeded SystemPrompt 'format' with ID: 1 and GUID: 7942b300-551b-4600-8aec-7aac2a145a91
Seeded SystemPrompt 'linting_system' with ID: 2 and GUID: 2b3f49ef-147c-4b28-8c92-20c38e5f5694
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.
Seeded agent provider configurations successfully.
✅ Seeded state provider configurations successfully.
Seeded system provider configurations successfully.
✅ Seeded controller providers
✅ Seeded program providers
✅ Seeded session configs
✅ Loaded environment variables from env/.env


In [2]:
from app.factories.program_provider_factory import ProgramProviderFactory

program = ProgramProviderFactory.create(id=1)
output = program.run({
    "session_id": "1",
    "file_name": "tests/example.py"
})

print("✅ Experiment completed")
print(output)

✅ Experiment completed
{'state': 'end', 'file_name': 'tests/example.py', 'working_file': 'tests\\example_working.py', 'session_id': '1', 'reason': 'preprocessing complete', '_last_state': 'preprocessing', 'output': {'state': 'end', 'file_path': 'tests\\example.py', 'working_file': 'tests\\example_working.py', 'file_name': 'tests/example.py', 'session_id': '1', 'reason': 'after preprocessing', '_last_state': 'preprocess'}}


In [3]:
from app.factories.agent_provider_factory import AgentProviderFactory

agent = AgentProviderFactory.create(id=2)  # The linting_generator_agent_provider
output = agent.run({
    "file_path": "tests/example.py",
    "system": "linting",
    "session_id": "1"
}, session_id="1")

print(output)


📸 Writing snapshot to: C:\Repos\codecritic\experiments\snapshots
[CODE]
def greet(name: str) -> str:
    return f"Hello, {name}!"
[/CODE]

[CONVERSATION_LOG_ENTRY]
The original code was already well-formatted and adhered to PEP8 guidelines. It included type hints and was clear and concise. No changes were necessary as the code was already optimal for its purpose.
[/CONVERSATION_LOG_ENTRY]


In [4]:
import json
import pandas as pd
from pathlib import Path

snapshots_root = Path("experiments/snapshots/1")
meta_files = list(snapshots_root.glob("*.meta.json"))
assert meta_files, "❌ No .meta.json files were created"

def load_metadata(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    data["snapshot_id"] = path.stem.replace(".meta", "")
    return data

metadata = [load_metadata(p) for p in meta_files]
df = pd.DataFrame(metadata)

required_fields = [
    "timestamp", "system", "agent", "score", "state", "decision",
    "line_count_before", "line_count_after",
    "function_count_before", "function_count_after",
    "symbol_count_before", "symbol_count_after",
    "branch_count_delta", "line_count_delta"
]

missing_fields = [field for field in required_fields if field not in df.columns]
assert not missing_fields, f"❌ Missing expected fields: {missing_fields}"

print("✅ Snapshot metadata verified successfully")
df[required_fields].head()

✅ Snapshot metadata verified successfully


,timestamp,system,agent,score,state,decision,line_count_before,line_count_after,function_count_before,function_count_after,symbol_count_before,symbol_count_after,branch_count_delta,line_count_delta
0,2025-05-30T16:36:53.457862+00:00,linting,linting_generator_agent_provider,0.955,unknown,unknown,2,2,1,1,1,1,0,0


In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

# Path to your SQLite database
db_path = Path("experiments/codecritic.sqlite3")

# Connect and run query
conn = sqlite3.connect(db_path)
query = """
SELECT
    timestamp,
    session_id,
    snapshot_id,
    system,
    agent,
    score,
    state,
    decision,
    line_count_before,
    line_count_after,
    line_count_delta
FROM snapshot_metrics
ORDER BY timestamp DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
conn.close()

# Display results
print("✅ Latest snapshot metrics from database:")
df


✅ Latest snapshot metrics from database:


,timestamp,session_id,snapshot_id,system,agent,score,state,decision,line_count_before,line_count_after,line_count_delta
0,2025-05-30 16:36:53.457862,1,be578b59-a371-4e7b-b2a6-514920bab8e9,linting,linting_generator_agent_provider,0.955,unknown,unknown,2,2,0
